In [0]:
# Databricks notebook source
from pyspark.sql import functions as F, Window

dbutils.widgets.text("bronze_catalog", "bronze_germantelekom")
dbutils.widgets.text("silver_catalog", "silver_germantelekom")
dbutils.widgets.text("schema", "dev_customer")
dbutils.widgets.text("source_file_name", "")          # passed by the bronze task

bronze_catalog = dbutils.widgets.get("bronze_catalog")
silver_catalog = dbutils.widgets.get("silver_catalog")
schema         = dbutils.widgets.get("schema")
file_name      = dbutils.widgets.get("source_file_name").strip().strip("'\"")
assert file_name, "source_file_name is required (it comes from the bronze task)"

bronze_tbl     = f"{bronze_catalog}.{schema}.customer_data"
silver_tbl     = f"{silver_catalog}.{schema}.customer_data"
quarantine_tbl = f"{silver_catalog}.{schema}.customer_data_quarantine"
safe_name      = file_name.replace("'", "''")         # safe to use inside SQL text

def publish(**values):
    """Hand small values to the next tasks of the job (does nothing outside a job)."""
    for k, v in values.items():
        try:
            dbutils.jobs.taskValues.set(key=k, value=v)
        except Exception:
            print(f"Not running in a job, task value '{k}' not set")

print("Batch :", file_name)

# COMMAND ----------

# 1. Read ONLY this batch from bronze
batch = spark.table(bronze_tbl).filter(F.col("source_file_name") == file_name).cache()
in_n = batch.count()
if in_n == 0:
    dbutils.notebook.exit(f"No rows in bronze for {file_name}")
batch_ts = batch.agg(F.max("load_timestamp")).first()[0]      # time of this batch, used for the SCD2 dates
print("Rows in batch:", in_n)

# COMMAND ----------

# 2. Standardize: raw columns stay untouched (quarantine keeps what arrived), cleaned values go to c_* columns
prod_txt = F.lower(F.trim("installed_product"))
name_txt = F.initcap(F.regexp_replace(F.trim("customer_name"), r"\s+", " "))
spd_txt  = F.regexp_extract("speed", r"(\d+)", 1)
spd_prod = F.regexp_extract(prod_txt, r"-(\d+)", 1)           # vdsl-100mbps -> 100
seg_txt  = F.upper(F.trim("segment"))
tech_txt = F.regexp_extract(prod_txt, r"^(svvdsl|vdsl|adsl)", 1)

std = (batch
    .withColumn("c_id", F.upper(F.trim("customer_id")))
    .withColumn("c_name", F.when(name_txt.isNull() | (name_txt == ""), F.lit("Unknown")).otherwise(name_txt))
    .withColumn("c_pin", F.regexp_replace(F.trim("pincode"), r"(?i)^D-", ""))
    .withColumn("c_product", F.when(tech_txt != "", tech_txt))                # vdsl-100mbps -> vdsl, fiber -> NULL
    .withColumn("c_segment", F.when(seg_txt == "RTL", "RETAIL")
                              .when(seg_txt == "WSL", "WHOLESALE")
                              .otherwise(seg_txt))
    # speed: from the speed column (negative values rejected), else from the product text
    .withColumn("spd_col", F.when(F.trim("speed").rlike(r"^-"), F.lit(None))
                            .when(spd_txt != "", spd_txt.cast("int")))
    .withColumn("spd_prod", F.when(spd_prod != "", spd_prod.cast("int")))
    .withColumn("c_speed", F.coalesce("spd_col", "spd_prod")))

# COMMAND ----------

# 3. Row-level checks: one boolean per rule (a NULL result counts as failed)
def ok(cond):
    return F.coalesce(cond, F.lit(False))

valid_pair = F.expr("""
  (c_product = 'adsl'   AND c_speed IN (16, 25)) OR
  (c_product = 'vdsl'   AND c_speed IN (50, 100, 250)) OR
  (c_product = 'svvdsl' AND c_speed IN (100, 250))""")

chk = (std
    .withColumn("chk_id",      ok(F.col("c_id").rlike(r"^C\d{7}$")))
    .withColumn("chk_pin",     ok(F.col("c_pin").rlike(r"^\d{5}$")))
    .withColumn("chk_segment", ok(F.col("c_segment").isin("RETAIL", "WHOLESALE")))
    .withColumn("chk_product", ok(F.col("c_product").isNotNull()))
    .withColumn("chk_speed",   ok(F.col("c_speed") > 0))
    .withColumn("chk_pair",    ok(valid_pair)))

flagged = (chk
    .withColumn("is_valid", F.expr("chk_id AND chk_pin AND chk_segment AND chk_product AND chk_speed AND chk_pair"))
    .withColumn("reject_reason", F.concat_ws(", ",
        F.when(~F.col("chk_id"),      F.lit("invalid customer_id")),
        F.when(~F.col("chk_pin"),     F.lit("invalid pincode")),
        F.when(~F.col("chk_segment"), F.lit("invalid segment")),
        F.when(~F.col("chk_product"), F.lit("invalid product")),
        F.when(~F.col("chk_speed"),   F.lit("invalid speed")),
        F.when(F.col("chk_product") & F.col("chk_speed") & ~F.col("chk_pair"), F.lit("product/speed mismatch")))))

# COMMAND ----------

# 4. One row per customer in this batch (the extra rows become "duplicate")
w = Window.partitionBy("c_id").orderBy(F.col("c_speed").desc(), F.col("c_name"))
ranked = flagged.filter("is_valid").withColumn("rn", F.row_number().over(w))

good = ranked.filter("rn = 1").cache()
dups = ranked.filter("rn > 1")
bad  = flagged.filter("NOT is_valid")

q_cols = ["customer_id", "customer_name", "pincode", "installed_product", "speed", "segment",
          "reject_reason", "corrupt_record", "source_file_name", "load_timestamp"]

bad_q  = bad.withColumn("corrupt_record", F.lit(None).cast("string")).select(*q_cols)
dups_q = (dups.withColumn("reject_reason", F.lit("duplicate customer_id in batch"))
              .withColumn("corrupt_record", F.lit(None).cast("string")).select(*q_cols))
quarantine_df = bad_q.unionByName(dups_q).cache()

good_n, quarantine_n = good.count(), quarantine_df.count()
print(f"in={in_n} | valid={good_n} | quarantined={quarantine_n}")

# COMMAND ----------

# 5. Reconcile BEFORE writing: every bronze row of this batch must land in exactly one place
if in_n != good_n + quarantine_n:
    raise Exception(f"Silver reconciliation failed for {file_name}: {in_n} != {good_n} + {quarantine_n}")

# COMMAND ----------

# 6. Quarantine (delete this batch first, so a retry after a failure cannot duplicate rows)
spark.sql(f"DELETE FROM {quarantine_tbl} WHERE source_file_name = '{safe_name}'")
quarantine_df.write.format("delta").mode("append").saveAsTable(quarantine_tbl)

display(quarantine_df.groupBy("reject_reason").count().orderBy(F.col("count").desc()))

# COMMAND ----------

# 7. SCD2 MERGE (business key: customer_id; change detection: tracked columns compared null-safe)
(good.select(F.col("c_id").alias("customer_id"), F.col("c_name").alias("customer_name"),
             F.col("c_pin").alias("pincode"), F.col("c_product").alias("installed_product"),
             F.col("c_speed").alias("speed_mbps"), F.col("c_segment").alias("segment"))
     .withColumn("effective_start_date", F.lit(batch_ts))
     .createOrReplaceTempView("updates"))

changed = """(NOT (u.customer_name <=> c.customer_name) OR NOT (u.pincode <=> c.pincode)
           OR NOT (u.installed_product <=> c.installed_product) OR NOT (u.speed_mbps <=> c.speed_mbps)
           OR NOT (u.segment <=> c.segment))"""

spark.sql(f"""
MERGE INTO {silver_tbl} AS t
USING (
  -- changed customers: this copy matches the old current row so it can be closed
  SELECT u.customer_id AS merge_key, u.*
  FROM updates u JOIN {silver_tbl} c ON u.customer_id = c.customer_id AND c.is_current = true
  WHERE {changed}
  UNION ALL
  -- new customers, and the new version of changed customers: these are inserted
  SELECT CAST(NULL AS STRING) AS merge_key, u.*
  FROM updates u LEFT JOIN {silver_tbl} c ON u.customer_id = c.customer_id AND c.is_current = true
  WHERE c.customer_id IS NULL OR {changed}
) AS s
ON t.customer_id = s.merge_key AND t.is_current = true
WHEN MATCHED THEN UPDATE SET is_current = false, effective_end_date = s.effective_start_date
WHEN NOT MATCHED THEN INSERT (customer_id, customer_name, pincode, installed_product, speed_mbps,
                              segment, effective_start_date, effective_end_date, is_current)
  VALUES (s.customer_id, s.customer_name, s.pincode, s.installed_product, s.speed_mbps,
          s.segment, s.effective_start_date, NULL, true)
""")

# COMMAND ----------

# 8. Check AFTER the merge
multi_current = (spark.table(silver_tbl).filter("is_current = true")
                 .groupBy("customer_id").count().filter("count > 1").count())
if multi_current > 0:
    raise Exception(f"{multi_current} customers have more than one current row")

current_for_batch = (spark.table(silver_tbl).filter("is_current = true")
                     .join(good.select(F.col("c_id").alias("customer_id")), "customer_id").count())
if current_for_batch != good_n:
    raise Exception(f"Post-merge check failed: {good_n} customers merged, {current_for_batch} current rows found")

# COMMAND ----------

# 9. Hand the counts to the final log task
publish(silver_in_rows=int(in_n), silver_valid_rows=int(good_n), silver_quarantined_rows=int(quarantine_n))
print(f"Silver done for {file_name}: in={in_n}, merged={good_n}, quarantined={quarantine_n}")